In [ ]:
import pandas as pd
from pyfaidx import Fasta
from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from Bio.Seq import Seq

In [ ]:
celline_arrary_df = pd.read_csv("BRCA_celline_circRNA_arrary.csv")
celline_arrary_df

In [ ]:
from Bio import SeqIO

fasta_file = "../../resources/circRNA/hsa_circbase_seq.fa"
fasta_dict = {record.id.split('|')[0]: str(record.seq) for record in SeqIO.parse(fasta_file, "fasta")}

celline_arrary_df['sequence'] = celline_arrary_df['CircRNA'].map(fasta_dict)

celline_arrary_df

In [ ]:
celline_arrary_df['sequence'].isnull().sum()

In [ ]:
missing_seq_mask = celline_arrary_df['sequence'].isna()
missing_df = celline_arrary_df[missing_seq_mask].copy()

print(f"Initial records with missing sequences: {len(missing_df)}")

if not missing_df.empty:
    missing_df['CircStart'] = missing_df['CircStart'].astype(int)
    missing_df['CircEnd'] = missing_df['CircEnd'].astype(int)

    bed_df = pd.read_csv('../../resources/circAtlas/human_bed_v3.0.txt', sep='\t')
    bed_df['Start'] = bed_df['Start'].astype(int)
    bed_df['End'] = bed_df['End'].astype(int)
    
    print(f"missing_df chromosome-format example: {missing_df['Chromosome'].iloc[0] if len(missing_df) > 0 else 'N/A'}")
    print(f"bed_df chromosome-format example: {bed_df['Chro'].iloc[0] if len(bed_df) > 0 else 'N/A'}")

    matched_chunks = []
    for d_start in [-1, 0, 1]:
        for d_end in [-1, 0, 1]:
            temp_missing = missing_df.copy()
            temp_missing['Start_match'] = temp_missing['CircStart'] + d_start
            temp_missing['End_match'] = temp_missing['CircEnd'] + d_end
            
            chunk = pd.merge(
                temp_missing[['CircRNA', 'Chromosome', 'Strand', 'Start_match', 'End_match']], 
                bed_df[['Chro', 'Strand', 'Start', 'End', 'circAltas_ID']], 
                left_on=['Chromosome', 'Strand', 'Start_match', 'End_match'], 
                right_on=['Chro', 'Strand', 'Start', 'End'],
                how='inner'
            )
            matched_chunks.append(chunk)

    if matched_chunks:
        matched_df = pd.concat(matched_chunks, ignore_index=True)
        matched_df = matched_df.drop_duplicates(subset=['CircRNA', 'circAltas_ID'])
        
        print(f"Coordinate-matched records after deduplication: {len(matched_df)}")
        
        if not matched_df.empty:
            seq_df = pd.read_csv('../../resources/circAtlas/human_sequence_v3.0', sep=' ', header=None, names=['circAltas_ID', 'Sequence'])
            
            merged_seq = pd.merge(matched_df, seq_df, on='circAltas_ID', how='left')
            
            valid_seq_count = merged_seq['Sequence'].notna().sum()
            print(f"Records with sequences retrieved by circAltas_ID: {valid_seq_count}")
            
            seq_agg_dict = merged_seq.groupby('CircRNA')['Sequence'].apply(lambda x: ','.join(x.dropna().astype(str))).to_dict()
            
            print(f"Unique circRNAs prepared for backfilling: {len(seq_agg_dict)}")

            celline_arrary_df.loc[missing_seq_mask, 'sequence'] = celline_arrary_df.loc[missing_seq_mask, 'CircRNA'].map(seq_agg_dict)
    else:
        print("Coordinate matching was skipped because no matching block was available.")

print(f"Records still missing sequences after backfilling: {celline_arrary_df['sequence'].isna().sum()}")

In [ ]:
filtered_df = celline_arrary_df[celline_arrary_df['sequence'].notna()].copy()


# filtered_df.to_csv("./filtered_celline_arrary_df.csv", index=False)
filtered_df = filtered_df[filtered_df['sequence'].str.len() <= 10000]

filtered_df = filtered_df[~filtered_df['sequence'].str.contains('N', case=False, na=False)]

In [ ]:
columns_to_save = ["CircRNA", "MDA-MB-231_original", "Spliced seq length", "Sequence"]
filtered_df.to_csv("./filtered_celline_arrary_df.csv", index=False)